# ML-04 — Search Intelligence Data Contract

This notebook defines the data contract for **Core Lane 2: Refresh & Opportunity Scoring**. It details the unit of analysis, time windows, field classifications, verified statistics from queries, and observed data limits.

## 1. Unit of analysis + time window

### Unit of Analysis (The Grain)
One row in our modeled dataset represents **one pseudonymized content item (page) for a given client at a given decision moment $t$**.
- Unique Key: `client_hash_id` × `content_hash_id` × `snapshot_date`.
- In a static snapshot (like the starter dataset `content_refresh_anonymized.csv`), this maps 1-to-1 with a unique `content_id` because it represents a single moment in time.

### Time Windows
To map this to a supervised learning task (predicting persistent traffic decline), we partition the time series into distinct, non-overlapping windows:
1. **Feature Window**: $[t - 59 \text{ days}, t]$ (60-day history). All features (traffic levels, CTR, position averages, engagement rates) are calculated strictly within this window. This is further split into:
   - Recent 30 days: $[t - 29, t]$
   - Previous 30 days: $[t - 59, t - 30]$
2. **Outcome (Label) Window**: $[t + 1 \text{ day}, t + 30 \text{ days}]$ (30-day future). This window is used to measure future impressions and sessions to determine the target label.

This separation ensures **zero data leakage**: no information from the outcome window is ever used to construct model features.

In [1]:
import duckdb
import os

# Retrieve token from the environment variable (or fall back if set)
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN environment variable is not set. Please set it to proceed.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET enable_server_cert_verification = false;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':       f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':   f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully and registered tables.")

Connected successfully and registered tables.


## 2. Fields: feature / label / context / excluded

Every field we interact with in the dataset falls into one of four mutually exclusive buckets:

### A. Feature Fields (Knowable at or before decision moment $t$)
- **GSC Search Performance**: `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (aggregated over recent/previous 30-day windows).
- **GA4 User Engagement**: `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `scroll_events`, `sessions_ai` (aggregated over recent/previous 30-day windows).
- **Query-Mix Characteristics (from `fact_query_90d` over the historical window)**:
  - `content_visible_query_count`: The number of distinct queries the page ranks for.
  - `rare_impressions_share`: Share of impressions from low-volume keywords.
  - `anonymized_impressions_share`: Share of impressions anonymized by Google.
  - `top_query_share`: Concentration of impressions on the top query (derived as `top_query_impressions / kept_impressions`).
- **Content Properties**: `word_count`, `char_count`, `content_type`, `main_intent`.
- **Keyword Context**: `search_volume`, `competition`, `competition_level`, `cpc`.
- **Lifecycle Metrics**: `content_age_days` (days since creation), `days_since_last_update`.

### B. Label / Proxy Fields (Representing the future outcome)
- `is_declining_label`: The target binary classification label. Evaluated on the outcome window $[t+1, t+30]$.
- `trend_direction` and `trend_pct` calculated in the outcome window: Excluded from features because they represent the label itself (leakage).

### C. Context Fields (For joining, splitting, and grouping; never used as model inputs)
- `content_hash_id` / `content_id`: Page identifier.
- `client_hash_id` / `client_id`: Client identifier. Crucial for **client-holdout splits** in cross-validation.
- `report_date` (or $t$): Snapshot date.
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`: Data validity flags.

### D. Excluded Fields (Not relevant for prediction, or future information)
- `provider_used` / `model_used`: Excluded because LLM choice at generation time is metadata, not a predictive feature of search performance decay.
- `client_created_date` / `client_updated_date`: Administrative database timestamps.

In [2]:
print("dim_content schema columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()[['column_name', 'column_type']].to_string(index=False))

dim_content schema columns:


               column_name column_type
            client_hash_id     VARCHAR
           content_hash_id     VARCHAR
           keyword_hash_id     VARCHAR
               url_hash_id     VARCHAR
        keyword_char_count      BIGINT
       keyword_token_count      BIGINT
            url_char_count      BIGINT
      content_created_date        DATE
      content_updated_date        DATE
              content_type     VARCHAR
             search_volume      BIGINT
               competition      DOUBLE
         competition_level     VARCHAR
                       cpc      DOUBLE
               main_intent     VARCHAR
                 backlinks      BIGINT
            category_count      BIGINT
      keyword_created_date        DATE
             provider_used     VARCHAR
                model_used     VARCHAR
                char_count      BIGINT
                word_count      BIGINT
       last_optimized_date        DATE
optimization_eligible_date        DATE
              is_publishe

## 3. Verify it with queries (grain, counts, missing values, windows)

Here we verify every contract claim with executed SQL queries.

### A. Table Row Counts
We confirm the size of our dimensional and sample fact tables.

In [3]:
for name in ['dim_clients', 'dim_content', 'fact_daily_sample', 'fact_query_90d']:
    src = TABLES[name]
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f"{name:20} {n:>12,} rows")

dim_clients                   104 rows


dim_content               519,606 rows


fact_daily_sample      11,694,072 rows


fact_query_90d          2,414,248 rows


### B. Grain Check (Deduplication)
For the grain to hold, `report_date` × `client_hash_id` × `content_hash_id` should uniquely identify a row in our performance facts.
As checked below, `fact_daily_sample` contains exactly 6,390 duplicate rows (0.05% of the total dataset), which represents a minor data integration duplication. We handle this by deduplicating using `MAX` aggregation in our features.
For `fact_query_90d`, the grain `client_hash_id` × `content_hash_id` × `query_hash_id` is verified to have 0 duplicates.

In [4]:
dup_daily = con.sql(f"""
    WITH counts AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
        FROM {TABLES['fact_daily_sample']}
        GROUP BY 1, 2, 3
    )
    SELECT SUM(CASE WHEN c > 1 THEN 1 ELSE 0 END) AS duplicate_grains
    FROM counts
""").fetchone()[0]
print(f"Duplicate grains in daily performance sample: {dup_daily}")

dup_query = con.sql(f"""
    WITH counts AS (
        SELECT client_hash_id, content_hash_id, query_hash_id, COUNT(*) c
        FROM {TABLES['fact_query_90d']}
        GROUP BY 1, 2, 3
    )
    SELECT SUM(CASE WHEN c > 1 THEN 1 ELSE 0 END) AS duplicate_grains
    FROM counts
""").fetchone()[0]
print(f"Duplicate grains in query table: {dup_query}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grains in daily performance sample: 6390


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grains in query table: 0


### C. Missingness Rates and Structured Gaps
We verify missing value rates. The data-use guidelines state that missingness follows the `content_type`.
As queried below:
- `feedly article` has exactly **100% missing search_volume** (since RSS feed content is not mapped to target keywords).
- `keyword article` has **18.6% missing search_volume** and **38.1% missing word_count**.
- `comparison article` has almost no missing values (~0.1%).

This demonstrates that blindly imputing missing values with 0 or a mean would introduce a strong categorical proxy signal. Instead, we must represent missingness using indicator flags.

In [5]:
df_missing = con.sql(f"""
    SELECT 
        content_type, 
        COUNT(*) AS total_rows,
        ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS pct_missing_search_volume,
        ROUND(AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS pct_missing_word_count
    FROM {TABLES['dim_content']}
    GROUP BY content_type
""").df()
print(df_missing.to_string(index=False))

      content_type  total_rows  pct_missing_search_volume  pct_missing_word_count
   keyword article      459174                      18.64                   38.08
comparison article        3408                       0.09                    0.09
    feedly article       57024                     100.00                    5.14


### D. Time Windows Verification (Unbalanced Panel)
We query the start and end dates of search history for different clients to verify that the panel is unbalanced.

In [6]:
clients_history = con.sql(f"""
    SELECT 
        client_hash_id,
        gsc_data_start,
        ga4_data_start
    FROM {TABLES['dim_clients']}
    WHERE gsc_data_start IS NOT NULL
    ORDER BY gsc_data_start ASC
    LIMIT 10
""").df()
print(clients_history.to_string(index=False))

         client_hash_id gsc_data_start ga4_data_start
client_9958f0a7ae1df715     2025-01-27     2025-10-29
client_ff644d8251367cbb     2025-01-27     2025-10-29
client_73cda7b4e4f265ea     2025-02-11     2026-03-24
client_fef1a8f436438636     2025-03-11     2026-03-06
client_62f4a7e64f5e0096     2025-06-07            NaT
client_b10cb2997d0c7c86     2025-06-18     2025-11-15
client_65de48885f4ef01b     2025-06-21     2026-02-19
client_c182d11e4862a37d     2025-06-21     2026-02-20
client_3197e6291363b4db     2025-06-29     2025-11-09
client_625b6439094e23e4     2025-07-01     2026-02-19


## 4. Data limits

Every dataset has boundaries where it ceases to be reliable. We document the key limitations of the search intelligence dataset:

1. **Unbalanced Panel History**: The search data start dates (`gsc_data_start`) differ wildly per client. We cannot apply a global calendar-time window across all clients without truncating history. We must evaluate performance relative to each client's specific history depth.
2. **GSC-only Early History**: GA4 tracking (`ga4_data_start`) starts later than GSC for many clients. Rows before GA4 start are zero-filled with `ga4_data_available = FALSE`. Model features utilizing GA4 metrics must filter out rows where `ga4_data_available` is False, otherwise the model will interpret lack of tracking as zero engagement.
3. **Tail Query Anonymization**: Google Search Console anonymizes search terms with low click volume to protect user privacy. In `fact_query_90d`, this is represented by `anonymized_impressions_share`, which limits our ability to audit specific queries for tail pages.
4. **Content Registration Lag**: When a content item is registered mid-window, the daily facts only capture performance from the registration date forward, while the query table attributes search performance for the full 90 days. We must align windows carefully and handle registration date mismatch.

In [7]:
ga4_check = con.sql(f"""
    SELECT 
        ga4_data_available,
        COUNT(*) AS total_rows,
        AVG(ga4_pageviews) AS avg_pageviews,
        AVG(ga4_sessions) AS avg_sessions
    FROM {TABLES['fact_daily_sample']}
    GROUP BY ga4_data_available
""").df()
print(ga4_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 ga4_data_available  total_rows  avg_pageviews  avg_sessions
               <NA>     2397428            NaN           NaN
              False     8651918       0.000000      0.000000
               True      644726       7.539787      4.280521


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.